# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [15]:
df.columns.to_list()


['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [16]:
df.shape

(1338, 7)

In [17]:
df.info

<bound method DataFrame.info of       age     sex     bmi  children smoker     region      charges
0      19  female  27.900         0    yes  southwest  16884.92400
1      18    male  33.770         1     no  southeast   1725.55230
2      28    male  33.000         3     no  southeast   4449.46200
3      33    male  22.705         0     no  northwest  21984.47061
4      32    male  28.880         0     no  northwest   3866.85520
...   ...     ...     ...       ...    ...        ...          ...
1333   50    male  30.970         3     no  northwest  10600.54830
1334   18  female  31.920         0     no  northeast   2205.98080
1335   18  female  36.850         0     no  southeast   1629.83350
1336   21  female  25.800         0     no  southwest   2007.94500
1337   61  female  29.070         0    yes  northwest  29141.36030

[1338 rows x 7 columns]>

## A.2. Missing values & Duplicate data

In [22]:
# TODO
df.duplicated().sum()
df=df.drop_duplicates()

In [20]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

## A.3. Invalid values

In [23]:
df.describe()


,age,bmi,children,charges
count,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,1.095737,13279.121487
std,14.044333,6.100468,1.205571,12110.359656
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.290000,0.000000,4746.344000
50%,39.000000,30.400000,1.000000,9386.161300
75%,51.000000,34.700000,2.000000,16657.717450
max,64.000000,53.130000,5.000000,63770.428010


In [26]:
for col in ['sex', 'smoker', 'region']:print(col, df[col].unique())

sex <StringArray>
['female', 'male']
Length: 2, dtype: str
smoker <StringArray>
['yes', 'no']
Length: 2, dtype: str
region <StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [ ]:
df['bmi_group']=['Normal' if x <25  else  'Overweight' if x < 30 else "Obese" for x in df['bmi'] ]


In [29]:
df.head(2)

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.90,0,yes,southwest,16884.9240,Overweight
1,18,male,33.77,1,no,southeast,1725.5523,Obese


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [30]:

df['charges'].mean()

np.float64(13279.121486655948)

In [31]:
df['charges'].median()

np.float64(9386.1613)

In [33]:
df['charges'].mode()[0]

np.float64(1121.8739)

## Group 2 — Dispersion

In [34]:
df['charges'].max() - df['charges'].min()

np.float64(62648.554110000005)

In [35]:
df['charges'].quantile(0.75) - df['charges'].quantile(0.25)

np.float64(11911.37345)

In [36]:
df['charges'].var()

np.float64(146660811.0060086)

In [37]:
df['charges'].std()

np.float64(12110.359656344175)

## Group 3 — Location and Shape

In [42]:
df['charges'].quantile([0.25, 0.50, 0.75])


0.25     4746.34400
0.50     9386.16130
0.75    16657.71745
Name: charges, dtype: float64

In [43]:
df['charges'].skew()

np.float64(1.5153909108403483)

In [44]:
df['charges'].kurtosis()

np.float64(1.6042206849514362)

---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [46]:
df.head(2)


,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.90,0,yes,southwest,16884.9240,Overweight
1,18,male,33.77,1,no,southeast,1725.5523,Obese


In [53]:
# Cách 1: Dùng pivot_table (Ngắn gọn, trực quan nhất)
smoker_avg = df.pivot_table(index='region', columns='smoker', values='charges', aggfunc='mean')
smoker_avg['ratio'] = smoker_avg['yes'] / smoker_avg['no']
print(smoker_avg)

smoker              no           yes     ratio
region                                        
northeast  9165.531672  29673.536473  3.237514
northwest  8582.467101  30192.003182  3.517870
southeast  8032.216309  34844.996824  4.338155
southwest  8019.284513  32269.063494  4.023933


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [50]:
df.head(2)
 

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.90,0,yes,southwest,16884.9240,Overweight
1,18,male,33.77,1,no,southeast,1725.5523,Obese


In [54]:

total_bmi_smoker = df[df['smoker'] == 'yes']['bmi'].sum()
total_bmi_non_smoker = df[df['smoker'] == 'no']['bmi'].sum()
if total_bmi_smoker > total_bmi_non_smoker:
    print("-> Nhóm HÚT THUỐC có tổng BMI lớn hơn.")
elif total_bmi_smoker < total_bmi_non_smoker:
    print("-> Nhóm KHÔNG HÚT THUỐC có tổng BMI lớn hơn.")
else:
    print("-> Tổng BMI của 2 nhóm BẰNG NHAU.")

-> Nhóm KHÔNG HÚT THUỐC có tổng BMI lớn hơn.


## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [56]:
df.head(2)

,age,sex,bmi,children,smoker,region,charges,bmi_group
0,19,female,27.90,0,yes,southwest,16884.9240,Overweight
1,18,male,33.77,1,no,southeast,1725.5523,Obese


In [60]:

region_charges = df.pivot_table(
    index='region', 
    values='charges', 
    aggfunc=['mean', 'median', 'std']
)
region_charges.columns = ['Mean', 'Median', 'Std']
region_charges = region_charges.sort_values('Mean', ascending=False)
print(region_charges)

                   Mean        Median           Std
region                                             
southeast  14735.411438   9294.131950  13971.098589
northeast  13406.384516  10057.652025  11255.803066
northwest  12450.840844   8976.977250  11073.125699
southwest  12346.937377   8798.593000  11557.179101


## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [61]:
children_charges = df.pivot_table(
    index='children', 
    values='charges', 
    aggfunc=['count', 'mean', 'median']
)

# Đổi tên cột cho dễ nhìn
children_charges.columns = ['Count', 'Mean', 'Median']
print(children_charges)

          Count          Mean       Median
children                                  
0           573  12384.695344   9863.47180
1           324  12731.171832   8483.87015
2           240  15073.563734   9264.97915
3           157  15355.318367  10600.54830
4            25  13850.656311  11033.66170
5            18   8786.035247   8589.56505


## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [62]:

corr_age = df['age'].corr(df['charges'])
print(f"Correlation between Age and Charges: {corr_age:.4f}")
if corr_age > 0.7:
    print("-> Tuổi và Chi phí có mối tương quan RẤT MẠNH.")
elif corr_age > 0.3:
    print("-> Tuổi và Chi phí có mối tương quan TRUNG BÌNH (Tuổi tăng thì chi phí có xu hướng tăng).")
else:
    print("-> Mối tương quan giữa Tuổi và Chi phí YẾU hoặc không đáng kể.")

Correlation between Age and Charges: 0.2983
-> Mối tương quan giữa Tuổi và Chi phí YẾU hoặc không đáng kể.


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Chi phí bảo hiểm y tế chịu ảnh hưởng mạnh nhất bởi thói quen hút thuốc; nhóm hút thuốc phải trả chi phí trung bình cao gấp gần 4 lần nhóm không hút. Tác động của BMI đến chi phí tăng vọt đặc biệt ở nhóm hút thuốc (BMI >= 30 kết hợp hút thuốc dẫn đến chi phí cao nhất). Tuổi tác có tương quan thuận rõ rệt với chi phí bảo hiểm. Trong khi đó, các yếu tố như khu vực địa lý hay số lượng con cái chỉ tạo ra sự chênh lệch nhỏ không đáng kể.  